<a href="https://colab.research.google.com/github/vMigliorini/pairRDD-operations/blob/main/PySparkAula2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pyspark
from pyspark.sql import SparkSession

In [6]:
spark = SparkSession.builder.appName("airports").getOrCreate()

In [7]:
sc = spark.sparkContext

In [8]:
rdd = sc.textFile("Airports.csv")
rdd.take(5)

['icao_code,iata_code,name,city,country,lat_deg,lat_min,lat_sec,lat_dir,lon_deg,lon_min,lon_sec,lon_dir,altitude,lat_decimal,lon_decimal,id',
 'AYGA,GKA,GOROKA,GOROKA,PAPUA NEW GUINEA,6,4,54,S,145,23,30,E,1610,-6.082,145.392,1',
 'AYLA,LAE,N/A,LAE,PAPUA NEW GUINEA,0,0,0,U,0,0,0,U,0,0,0,2',
 'AYMD,MAG,MADANG,MADANG,PAPUA NEW GUINEA,5,12,25,S,145,47,19,E,7,-5.207,145.789,3',
 'AYMH,HGU,MOUNT HAGEN,MOUNT HAGEN,PAPUA NEW GUINEA,5,49,34,S,144,17,46,E,1643,-5.826,144.296,4']

In [9]:
# Criacao do PairRDD no formato (airport name, city)
pairRDD = rdd.map(lambda x: (x.split(",")[2] , x.split(",")[3]))
pairRDD.take(5)

[('name', 'city'),
 ('GOROKA', 'GOROKA'),
 ('N/A', 'LAE'),
 ('MADANG', 'MADANG'),
 ('MOUNT HAGEN', 'MOUNT HAGEN')]

In [13]:
# retornar somente os registros em que o valor é PARIS
pairRDD_paris = pairRDD.filter(lambda x: x[1] == "PARIS")
pairRDD_paris.collect()

[('LE BOURGET', 'PARIS'),
 ('CHARLES DE GAULLE', 'PARIS'),
 ('N/A', 'PARIS'),
 ('ORLY', 'PARIS')]

In [18]:
# Capitalize somente os valores do PairRDD
pairRDD_capitalize = pairRDD_paris.mapValues(lambda x: x.capitalize())
pairRDD_capitalize_alt = pairRDD_paris.map(lambda x: (x[0], x[1].capitalize()))

pairRDD_capitalize.collect()

[('LE BOURGET', 'Paris'),
 ('CHARLES DE GAULLE', 'Paris'),
 ('N/A', 'Paris'),
 ('ORLY', 'Paris')]

In [20]:
# Ordenar os registros com base na chave
pairRDD_Paris_Sorted = pairRDD_capitalize.sortByKey()
pairRDD_Paris_Sorted_alt = pairRDD_capitalize.sortBy(lambda x: x[0])
pairRDD_Paris_Sorted.collect()

[('CHARLES DE GAULLE', 'Paris'),
 ('LE BOURGET', 'Paris'),
 ('N/A', 'Paris'),
 ('ORLY', 'Paris')]

In [53]:
# Teacher's challenge

# Converting airports to pairRDD
rdd_raw = sc.textFile("Airports.csv")
rdd_raw.take(5)

rdd_filter = rdd_raw.filter(lambda x: x.split(",")[0] != 'icao_code')

new_rdd = rdd_filter.map(lambda x: (x.split(",")[2], (x.split(",")[3] , int(x.split(",")[13]))))
new_rdd.take(5)

# Returning only lines with PARIS as its city name and with altitude > 10

rdd_paris_altitude = new_rdd.filter(lambda x: x[1][0] == "PARIS" and x[1][1] > 10)
rdd_paris_altitude.take(5)

# Capitalizing the city names

new_rdd_capitalize = rdd_paris_altitude.mapValues(lambda x: (x[0].capitalize(), x[1]))
new_rdd_capitalize.take(5)

# Sorting the registry by the altitude value, from highest to lowest

new_rdd_capitalize_sorted = new_rdd_capitalize.sortBy(lambda x: x[1][1], ascending=False)
new_rdd_capitalize_sorted.take(5)

[('CHARLES DE GAULLE', ('Paris', 120)),
 ('ORLY', ('Paris', 89)),
 ('LE BOURGET', ('Paris', 67))]

# Reimplementacao do WordCount usando ReduceByKey

In [58]:
rdd = sc.textFile("frankenstein.txt")
rdd.take(5)

['*** START OF THE PROJECT GUTENBERG EBOOK FRANKENSTEIN; OR, THE MODERN PROMETHEUS ***',
 '',
 'Frankenstein;',
 '',
 'or, the Modern Prometheus']

In [65]:
import re
rdd_flatten = rdd.flatMap(lambda x: re.findall(r"[a-z]+", x.lower()))
rdd_flatten.take(5)

['start', 'of', 'the', 'project', 'gutenberg']

In [67]:
# Criacao de um pairRDD

pairRDD = rdd_flatten.map(lambda x: (x,1))
pairRDD.take(5)

[('start', 1), ('of', 1), ('the', 1), ('project', 1), ('gutenberg', 1)]

In [68]:
# Remove as chaves com menos de duas letras

pairRDD = pairRDD.filter(lambda x: len(x[0]) > 2)
pairRDD.take(5)

[('start', 1), ('the', 1), ('project', 1), ('gutenberg', 1), ('ebook', 1)]

In [69]:
# Soma as ocorrências por chave
wordCount = pairRDD.reduceByKey(lambda x,y: x+y)
wordCount.take(5)

[('start', 1), ('mary', 1), ('shelley', 1), ('contents', 2), ('letter', 32)]

In [70]:
# Ordenar registros de acordo com a quantidade de ocorrências

wordCount_sorted = wordCount.sortBy(lambda x: x[1], ascending=False)
wordCount_sorted.take(5)

[('the', 4199), ('and', 2976), ('was', 1021), ('that', 1018), ('but', 687)]

Media de preco dos imoveis por numeros de quartos

In [86]:
# Teacher's Activity 1

# making the RDD and filtering the header
rdd = sc.textFile("RealEstate.csv")
rdd.take(5)

rdd_filter = rdd.filter(lambda x: x.split(",")[0] != 'MLS')
rdd_filter.take(5)

# Generating one pairRDD with this template: (bedrom quantity, (price, 1))
rdd_map = rdd_filter.map(lambda x: (int(x.split(",")[3] ), (float(x.split(",")[2]) , 1)))
rdd_map.take(5)

# reducing the bedroom values
rdd_reduced = rdd_map.reduceByKey(lambda x,y: (x[0]+y[0], x[1]+y[1]))
rdd_reduced.take(5)

# Getting the averages
rdd_average = rdd_reduced.mapValues(lambda x: round(x[0] / x[1], 2))
rdd_average.take(5)

# Sorting all the results
rdd_sorted = rdd_average.sortBy(lambda x: x[0])
rdd_sorted.collect()


[(0, 293450.0),
 (1, 169981.82),
 (2, 266356.37),
 (3, 359062.21),
 (4, 483475.65),
 (5, 657858.06),
 (6, 603225.0),
 (7, 325000.0),
 (10, 699000.0)]